In [ ]:
import pandas as pd
import numpy as np

In [ ]:
fernanda_df = pd.read_csv('teste2.csv')
fernanda_df.head()

In [ ]:
hernrique_df = pd.read_csv('teste.csv')
hernrique_df.head()

In [ ]:
print(fernanda_df.columns)

In [ ]:
print(hernrique_df.columns)

In [ ]:
print([col for col in hernrique_df.columns if "kmeans" in col])

In [ ]:
left_join = [col for col in hernrique_df.columns if col not in fernanda_df.columns]
right_join = [col for col in fernanda_df.columns if col not in hernrique_df.columns]
inner_join =[col for col in fernanda_df.columns if col in hernrique_df.columns]

print(f"Tamanho da intersecção: {len(inner_join)}, {inner_join}")
print(f"Tamanho do right_join: {len(right_join)}, {right_join}")
print(f"Tamanho do left_join: {len(left_join)}, {left_join}")

Conseguimos peceber que eu usei o nome das métricas no prefixo enquanto a Fernanda usou no prefixo

In [ ]:
metrics_name = ["max",
                "min",
                "mean",
                "median",
                "std",
                "var",
                "kurtois",
                "skewness",
                "attr_sparsity",
                "gmean",
                "hmean",
                "entropy",
                "prop_pca",
                "sparsity",
                "attr_sparsity",
                "iqr",
                "corr",
                "nrCorrAttr",
                "dbscan",
                "kmeans"
                
                ]



In [ ]:
right_join = [

      f"{parts[-1]}_{'_'.join(parts[:-1])}" if (parts := column_name.split('_')) and parts[-1] in metrics_name else column_name for column_name in right_join
]
right_join

In [ ]:
right_join = [col for col in right_join if col not in hernrique_df.columns]
print(f"Tamanho do right_join: {len(right_join)}, {right_join}")

Correlation no meu caso é corr...


In [ ]:
right_join = [ f"corr{item[11:]}" if item.startswith('correlation') else item for item in right_join]
right_join

In [ ]:
right_join = [col for col in right_join if col not in hernrique_df.columns]
print(f"Tamanho do right_join: {len(right_join)}, {right_join}")

<li>Fernanda chama de predict, eu chamo de prediction</li>
<li>uniqueness_ratio_ e nr_outliers ficaram de fora da conversão</li>
<li>kmeans intertia e kmenas n_clusters</li>
<li>compacness -> compactness</li>


DESCOBRI QUE NÃO ESTAVA COMPUTANDO OS KMEANS :D

In [ ]:
right_join = [col for col in right_join if not "predict" in col]
print(f"Tamanho do right_join: {len(right_join)}, {right_join}")

In [ ]:
print(len("nr_outliers"))
print(len("uniqueness_ratio"))


In [ ]:
right_join = [ f"uniqueness_ratio_{item[:-17]}" if item.endswith('uniqueness_ratio') else item for item in right_join]
right_join

In [ ]:
right_join = [ f"nr_outliers_{item[:-12]}" if item.endswith('nr_outliers') else item for item in right_join]
right_join

In [ ]:
right_join = [col  for col in right_join if col not in hernrique_df.columns]
right_join

F1-score são métricas targer e basline, tá tranquilo :D
Temos um guia de como converter as colunas de uma tabela em outra

In [ ]:

def transform_columns(name:str):
    parts = name.split("_")
    if(parts[-1] in metrics_name):
        name = f"{parts[-1]}_{'_'.join(parts[:-1])}"
    
    if(name.endswith("nr_outliers")):
        name = f"nr_outliers_{name[:-12]}"
    
    if(name.endswith("uniqueness_ratio")):
         name = f"uniqueness_ratio_{name[:-17]}"
    
    if(name=="compacness"):
        return "compactness"
    
    if("predict" in name):
        name = name.replace("predict","prediction")
    
    if(name.startswith('correlation')):
        name= f"corr{name[11:]}"
    return name

In [ ]:

fernanda_df.rename(columns=transform_columns,inplace=True)


In [ ]:
left_join = [col for col in hernrique_df.columns if col not in fernanda_df.columns]
right_join = [col for col in fernanda_df.columns if col not in hernrique_df.columns]
inner_join =[col for col in fernanda_df.columns if col in hernrique_df.columns]

print(f"Tamanho da intersecção: {len(inner_join)}, {inner_join}")
print(f"Tamanho do right_join: {len(right_join)}, {right_join}")
print(f"Tamanho do left_join: {len(left_join)}, {left_join}")


In [ ]:

fernanda_df.drop(columns=right_join,inplace=True)
hernrique_df.drop(columns=left_join,inplace=True)



In [ ]:
fernanda_df.sort_index(axis=1,inplace=True)
hernrique_df.sort_index(axis=1,inplace=True)

In [ ]:
hernrique_df.head()


In [ ]:
fernanda_df.head()

Esses valores coincidentes podem ter vindos todos da offline. É bom pegar linhas aleatórias

In [ ]:
print(f"Shape da Fernanda {fernanda_df.shape}")
print(f"Shape do Henrique {hernrique_df.shape}")

SUSPEITO

In [ ]:
print(fernanda_df.shape[0]-hernrique_df.shape[0])

In [ ]:
print(hernrique_df.iloc[1]==fernanda_df.iloc[1])
print(comp[comp == False].index.tolist())


In [ ]:
def has_diffs(pos)->bool:
    comp = hernrique_df.iloc[pos] == fernanda_df.iloc[pos]
    if(comp.all()):
         return True
    else:
        aut = comp[comp == False].index.tolist()
        print(len(aut),aut)
        return False

In [ ]:
has_diffs(1)

In [ ]:
print(hernrique_df["entropy_nswprice"])
print(fernanda_df["entropy_nswprice"][:1378])

Me parece aceitável...
Os predictions com certeza estariam diferentes por que os meta modelos utilizados não têm outptuna e os hiperparametros são elevemente diferentes. Os outros parecem ter diferenças de calores aceitáveis


Por estarem parecidos os datasets até a linha que eu tenho no henrique_df, acho que deve as linhas faltantes deveriam ter vindo do final da fase online, quando eu só dou update nos targets, ou algo do tipo

In [ ]:
comp =hernrique_df['uniqueness_ratio_nswprice']==fernanda_df["uniqueness_ratio_nswprice"]

print(comp)

In [ ]:
prim = comp.idxmin()
prim

Estamos computando algo diferente na fase online

Att: Descobri que troquei um > por um >=


Att: Os valores bateram


In [ ]:
diffs = (hernrique_df - fernanda_df).abs().max()
for col, valor in diffs.sort_values(ascending=False).items():
    print(f"{col}: {valor:.6f}")  

Diferença relevante = intertia, n_inter
Fernanda faz a normalização no kmeans, eu não